**Cleaning Data (Milestone 1) | Tyler Hilbert | April 2, 2026**

Now that the data has been dummied (See *Dummied Data.ipynb*), analysis of the data can commence. The purpose of this file is to review the data, as well as clean and add additional columns as needed. No visualizations are planned for this file, as it is just examining the file and brainstorming ideas.

Before getting into the file, my goals are as follows:
- Expand out the Registration Status and Academic Period codes
- Create a column with the Subject and Course combined
- Add the Intervention column

I'm sure other things will come up as I go through the process, but those are the minimum things that will help make this be a successful analysis.

*Note - any steps that don't "work" and break the file will be commented out, but still present. This way pressing the Run All button runs the file.*

**Importing Libraries and Looking at the data**

In [277]:
#Importing Libraries
import pandas as pd #importing pandas
import numpy as np #importing numpy

In [278]:
#Checking Out the Data Pt. 1 (Loading the data)
mtfull = pd.read_csv("mtgradesanon.csv") #Pulling in the mt grades file
mtfull.head() #running the head to make sure it works

,Unnamed: 0,Registration Status,Subject,Course,Campus,Final Grade,Mid Term Grade,Department,Major,Academic Period,Class
0,0,RW,AED,22860,KC,A-,B,ARCH,ID,202280,FR
1,1,RW,AED,22860,KC,A,A,ARCH,ID,202280,FR
2,2,RW,AED,22860,KC,A,A,ARCH,OTH,202280,JR
3,3,RW,AED,22860,KC,F,B,ARCH,ID,202280,SO
4,4,RW,AED,22860,KC,B,B,ARCH,ID,202280,FR


Now that the data is loaded in, I'm just going to go through the motions - checking out the shapes, keys, etc. Not much has changed from the original file, so I'm not expecting anything too shocking.

In [279]:
#Checking Out the Data Pt. 2 (Checking out the shape)
mtfull.shape #making sure the shape has come in successfully

(85341, 11)

In [280]:
#Checking Out the Data Pt. 3 (Checking out the Keys)
mtfull.keys() #checking out the keys

Index(['Unnamed: 0', 'Registration Status', 'Subject', 'Course', 'Campus',
       'Final Grade', 'Mid Term Grade', 'Department', 'Major',
       'Academic Period', 'Class'],
      dtype='object')

The Unnamed:0 is a new one - that will need changed since it doesn't make much sense as is.

In [281]:
#Checking Out the Data Pt. 4 (Checking out the Types)
mtfull.dtypes #checking out the types of the dataframe

Unnamed: 0              int64
Registration Status    object
Subject                object
Course                  int64
Campus                 object
Final Grade            object
Mid Term Grade         object
Department             object
Major                  object
Academic Period         int64
Class                  object
dtype: object

**Replacing A Column Header**

After reviewing the shape and types of data in the dataframe, the first thing I'd like to do is remove Unnamed: 0 as a column header. This is an unfortunate consequence of removing the ID in the dummying file. I think the best way to handle this is transforming the name of the column to "Record ID." In hindsight, I could have used the logic for shuffling the course numbers with the Student IDs, but I wanted to minimize the risk of anything getting out. 

In [282]:
#Replacing A Column Header
#Unnamed: 0 is an unfortunate consequence of removing the ID from the last round. I opted to replace it with an artificial ID column header - this step will be removed in the "final" version
mtfull = mtfull.rename(columns={"Unnamed: 0":"Record ID"}) #renaming the column to Record ID - not exact, but looks nicer

**Making a Course Code Column**

The first manipulation we are doing in this file is creating a Course Code column. In its current state, the Subject and Course # are separate columns - in many reports, both are presented together. I think adding a Course Code column will allow for two options for future filters:
- Filtering by the full course code - this may be useful in combining with the department value
- Filtering by subject *and then* course number. This may be useful for people honing in on a specific subject before a course.

In [283]:
#Making a Course Code Column Pt. 1 (Oops, it doesn't work)
#mtfull["Course Code"] = mtfull["Subject"] + " " + mtfull["Course"] #Attempting to combine them - the space is so they're not mushed together
#mtfull #Trying to run it - get an error because you cannot concatenate str and int

So that didn't work - the error I got was about trying to combine both a string and integer. I'm going to need to convert one of them. I'm going to do that with the course value. I think it's better than the subject, since course numbers are defined things and not a continuous number.

In [284]:
#Making a Course Code Column Pt. 2 (Converting Course to String)
mtfull["Course"] = mtfull["Course"].astype(str) #converting the course (int) to an object (str)
mtfull.dtypes #checking to make sure it worked

Record ID               int64
Registration Status    object
Subject                object
Course                 object
Campus                 object
Final Grade            object
Mid Term Grade         object
Department             object
Major                  object
Academic Period         int64
Class                  object
dtype: object

In [285]:
#Making a Course Code Column Pt. 3 (For Real this time)
mtfull["Course Code"] = mtfull["Subject"] + " " + mtfull["Course"] #Formally combining them - the space ensures there is a space between the two
mtfull #it worked this time!

,Record ID,Registration Status,Subject,Course,Campus,Final Grade,Mid Term Grade,Department,Major,Academic Period,Class,Course Code
0,0,RW,AED,22860,KC,A-,B,ARCH,ID,202280,FR,AED 22860
1,1,RW,AED,22860,KC,A,A,ARCH,ID,202280,FR,AED 22860
2,2,RW,AED,22860,KC,A,A,ARCH,OTH,202280,JR,AED 22860
3,3,RW,AED,22860,KC,F,B,ARCH,ID,202280,SO,AED 22860
4,4,RW,AED,22860,KC,B,B,ARCH,ID,202280,FR,AED 22860
...,...,...,...,...,...,...,...,...,...,...,...,...
85336,85336,RW,MDJ,20288,KC,B+,B,MDJ,FM,202310,SO,MDJ 20288
85337,85337,RW,MDJ,20288,KC,A,A,MDJ,FM,202310,JR,MDJ 20288
85338,85338,DD,MDJ,20288,KC,NaN,NaN,MDJ,COMM,202310,JR,MDJ 20288
85339,85339,RW,MDJ,20288,KC,B+,B+,MDJ,FM,202310,SO,MDJ 20288


Having this Course Code column will be handy when making buttons/dropdown menus later - less cumbersome than having to do subject first and then picking course numbers.

**Translating Status Codes**

The next step is moving to registration status codes. There are a lot of different codes in the system that mean the same thing, just impacted by timing/who does the thing. Writing them out would help explain to someone why certain grade combinations happen (i.e. why someone who dropped a course still has a final and midterm grade when they really shouldn't)

In [286]:
#Translating the Reg Status Codes Pt. 1 (Checking the Codes)
#There are a lot of codes in the system that mean the same thing, but with small differences (i.e. DD and W8 both mean drop, just when the drop happened)
regstatuscodes = mtfull["Registration Status"].unique() #Making a list of all the registration status codes in the file so I know what to change out
regstatuscodes #Generates a list of 15 codes that are present in the file

array(['RW', 'RE', 'WW', 'DD', 'W8', 'ND', 'R2', 'SF', 'B1', 'WD', 'NF',
       'RA', 'AW', 'DR', 'B5'], dtype=object)

When originally writing the below code, I just did Register, Withdraw, or Drop. I decided to get more specific because, like previously stated, there are certain grade combinations that do not make sense without understanding the registration status code.

Additionally, since we are doing a retroactive look at final grades, this specification will help us in adjusting the data to be more accurate as to what the grades would have looked like if we pulled them at the right time (i.e. right after midterm and/or final grades were posted).

In [287]:
#Translating the Reg Status Codes Pt. 2 (Making the Map)
regstatuscodestranslated = ["Registered","Std Withdrawn","Stopped Attending - Failed", "Admin Dropped","Std Dropped","Admin Dropped","Never Attended - Failed", "Registered", "Std Withdrawn", "Registered", "Admin Withdrawn", "Audited", "Admin Dropped", "Admin Withdrawn", "Std Dropped"]
regstatuscodestranslated

['Registered',
 'Std Withdrawn',
 'Stopped Attending - Failed',
 'Admin Dropped',
 'Std Dropped',
 'Admin Dropped',
 'Never Attended - Failed',
 'Registered',
 'Std Withdrawn',
 'Registered',
 'Admin Withdrawn',
 'Audited',
 'Admin Dropped',
 'Admin Withdrawn',
 'Std Dropped']

In [288]:
#Translating the Reg Status Codes Pt. 3 (Matching Codes to Full)
mapregstatuscode = dict(zip(regstatuscodes, regstatuscodestranslated)) #Making a dictionary that matches the codes to the full version
mapregstatuscode #running it so it works

{'RW': 'Registered',
 'RE': 'Std Withdrawn',
 'WW': 'Stopped Attending - Failed',
 'DD': 'Admin Dropped',
 'W8': 'Std Dropped',
 'ND': 'Admin Dropped',
 'R2': 'Never Attended - Failed',
 'SF': 'Registered',
 'B1': 'Std Withdrawn',
 'WD': 'Registered',
 'NF': 'Admin Withdrawn',
 'RA': 'Audited',
 'AW': 'Admin Dropped',
 'DR': 'Admin Withdrawn',
 'B5': 'Std Dropped'}

This method worked really well - I know we discussed manual mapping in class, but having an option for when there are a lot of items to be looked at is much more helpful!

In [289]:
#Translating the Reg Status Codes Pt. 4 (Replacing the Codes w/ Full)
mtfull["Registration Status"] = mtfull["Registration Status"].map(mapregstatuscode) #using the .map(), this is creating a new version of the dataframe with the full code.
mtfull #running to make sure it worked

,Record ID,Registration Status,Subject,Course,Campus,Final Grade,Mid Term Grade,Department,Major,Academic Period,Class,Course Code
0,0,Registered,AED,22860,KC,A-,B,ARCH,ID,202280,FR,AED 22860
1,1,Registered,AED,22860,KC,A,A,ARCH,ID,202280,FR,AED 22860
2,2,Registered,AED,22860,KC,A,A,ARCH,OTH,202280,JR,AED 22860
3,3,Registered,AED,22860,KC,F,B,ARCH,ID,202280,SO,AED 22860
4,4,Registered,AED,22860,KC,B,B,ARCH,ID,202280,FR,AED 22860
...,...,...,...,...,...,...,...,...,...,...,...,...
85336,85336,Registered,MDJ,20288,KC,B+,B,MDJ,FM,202310,SO,MDJ 20288
85337,85337,Registered,MDJ,20288,KC,A,A,MDJ,FM,202310,JR,MDJ 20288
85338,85338,Admin Dropped,MDJ,20288,KC,NaN,NaN,MDJ,COMM,202310,JR,MDJ 20288
85339,85339,Registered,MDJ,20288,KC,B+,B+,MDJ,FM,202310,SO,MDJ 20288


**Making a College Column**

Another column that I should make is a college column. Having a column by college will help create the visualizations later on in case someone wants to see the intervention rates or figures by college. I can foresee this process becoming very bulky if we seriously consider every major and blow this process up to every college. However, it would be doable (just a bit of frontend work of writing out every major...).

In [290]:
#Making a College Column Pt. 1 (Looking at Values)
mtfull["Major"].unique() #Checking out the values that are in the Major column

array(['ID', 'OTH', 'ARCH', 'FM', 'COMM', 'ARTH', 'FD', 'DNST', 'TDTP',
       'SART', 'COMA', 'DMP', 'VCD', 'JNL', 'EMAT', 'ADV', 'THEA', 'ARCS',
       'PR', 'ARTE', 'MUST', 'PHOT', 'MUS', 'MUED', 'MUT', 'APMD', 'UXDE',
       'DANC'], dtype=object)

In [291]:
#Making a College Column Pt. 2 (Making College Variables)
caedmajor = ["ID", "ARCH", "COMA", "ARCS"] #Majors in CAED
ccimajor = ["COMM", "DMP", "VCD", "JNL", "EMAT", "ADV", "PR", "PHOT", "APMD","UXDE"] #Majors in CCI
cotamajor = ["FM", "ARTH", "FD", "DNST", "TDTP", "SART", "THEA", "ARTE", "MUST", "MUS", "MUED", "MUT","DANC"] #Majors in CotA
othermajor = ["OTH"] #Majors outside those colleges

In [292]:
#Making a College Column Pt. 3 (Making Condition and Outcomes - Oops)
#collegecondition = [
#    (mtfull["Major"] = caedmajor), 
#    (mtfull["Major"] = ccimajor), 
#    (mtfull["Major"] = cotamajor), 
#    (mtfull["Other"] = othermajor)]
#outcomes = ["CAED", "CCI", "COTA", "OTH"]
#The issue seems to be tied to the = - going to try that

So when I wrote the above, I forgot to do the double == - thought that it would be a quick fix!

In [293]:
#Making a College Column Pt. 3 (Making Condition and Outcomes - Oops 2)
#colcond = [
#    (mtfull["Major"] == caedmajor), 
#    (mtfull["Major"] == ccimajor), 
#    (mtfull["Major"] == cotamajor), 
#    (mtfull["Other"] == othermajor)]
#colout = ["CAED", "CCI", "COTA", "OTH"]
#Nope - this doesn't work because there are too many values. So going to have to rethink this. Boo.

It wasn't - I got an error saying there were too many values, so I need to rethink the process. After some research and looking, the isin method is much better for this - having the program look for what list the major is in helps the program run much nicer.

In [294]:
#Making a College Column Pt. 4 (Making Condition and Outcomes - For Real This Time)
colcond = [ #So did some looking, and isin seems to be the way forward. Before was trying to find exact matches, this way is saying "Hey - check this list and see if it is there first"
    mtfull["Major"].isin(caedmajor),
    mtfull["Major"].isin(ccimajor),
    mtfull["Major"].isin(cotamajor),
    mtfull["Major"].isin(othermajor)] 

colout = ["CAED", "CCI", "CotA", "Other"]

In [295]:
#Making a College Column Pt. 5 (Making the Column - Oops)
#mtfull["College"] = np.select(colcond,colout) #forgot to have the "can't find column" - also realize now that I didn't need to make teh Other Major field

The above is just a typo - and like I said below, it makes me realize I didn't need to make an "Other" college variable. I could have put Other instead of Missing and it would've worked. However, it would be handy at catching those programs I missed, so making the other variable was handy.

In [296]:
#Making a College Column Pt. 6 (Making the Column - For real this time)
mtfull["College"] = np.select(colcond,colout, "Missing") #After thinking about it, I realized that it may be better to have a missing value - that way if I somehow missed something I can catch it
mtfull

,Record ID,Registration Status,Subject,Course,Campus,Final Grade,Mid Term Grade,Department,Major,Academic Period,Class,Course Code,College
0,0,Registered,AED,22860,KC,A-,B,ARCH,ID,202280,FR,AED 22860,CAED
1,1,Registered,AED,22860,KC,A,A,ARCH,ID,202280,FR,AED 22860,CAED
2,2,Registered,AED,22860,KC,A,A,ARCH,OTH,202280,JR,AED 22860,Other
3,3,Registered,AED,22860,KC,F,B,ARCH,ID,202280,SO,AED 22860,CAED
4,4,Registered,AED,22860,KC,B,B,ARCH,ID,202280,FR,AED 22860,CAED
...,...,...,...,...,...,...,...,...,...,...,...,...,...
85336,85336,Registered,MDJ,20288,KC,B+,B,MDJ,FM,202310,SO,MDJ 20288,CotA
85337,85337,Registered,MDJ,20288,KC,A,A,MDJ,FM,202310,JR,MDJ 20288,CotA
85338,85338,Admin Dropped,MDJ,20288,KC,NaN,NaN,MDJ,COMM,202310,JR,MDJ 20288,CCI
85339,85339,Registered,MDJ,20288,KC,B+,B+,MDJ,FM,202310,SO,MDJ 20288,CotA


In [297]:
#Making a College Column Pt. 7 (Checking for missing values)
collegecheck = mtfull.groupby("College").count()["Record ID"]
collegecheck

College
CAED     11366
CCI      15236
CotA     32185
Other    26554
Name: Record ID, dtype: int64

**Writing out the terms**

Quick tangent - I remember the first time I used the standard term codes when talking to others. They were confused by what it meant, as they were unfamiliar with the terms. Ever since, I try to always write out the full term when possible. This way anyone who does not understand the term codes can understand what is happening and track changes over time.

Below is going through the same process like we did with the registration status codes. Since not much changes, there's not much narrative to add.

In [298]:
#Making a Term Column Pt. 1 (Identifying periods)
shortterm = mtfull["Academic Period"].unique()
shortterm

array([202280, 202310, 202380, 202410, 202480, 202510, 202580, 202610])

In [299]:
#Making a Term Column Pt. 2 (Defining the periods)
fullterm = ["Fall 2022", "Spring 2023", "Fall 2023", "Spring 2024", "Fall 2024", "Spring 2025", "Fall 2025", "Spring 2026"]
fullterm

['Fall 2022',
 'Spring 2023',
 'Fall 2023',
 'Spring 2024',
 'Fall 2024',
 'Spring 2025',
 'Fall 2025',
 'Spring 2026']

In [300]:
#Making a Term Column Pt. 3 (Making the term map)
mapterm = dict(zip(shortterm, fullterm))
mapterm

{np.int64(202280): 'Fall 2022',
 np.int64(202310): 'Spring 2023',
 np.int64(202380): 'Fall 2023',
 np.int64(202410): 'Spring 2024',
 np.int64(202480): 'Fall 2024',
 np.int64(202510): 'Spring 2025',
 np.int64(202580): 'Fall 2025',
 np.int64(202610): 'Spring 2026'}

In [301]:
#Making a Term Column Pt. 4 (Overwriting the Academic Period column)
mtfull["Academic Period"] = mtfull["Academic Period"].map(mapterm)
mtfull

,Record ID,Registration Status,Subject,Course,Campus,Final Grade,Mid Term Grade,Department,Major,Academic Period,Class,Course Code,College
0,0,Registered,AED,22860,KC,A-,B,ARCH,ID,Fall 2022,FR,AED 22860,CAED
1,1,Registered,AED,22860,KC,A,A,ARCH,ID,Fall 2022,FR,AED 22860,CAED
2,2,Registered,AED,22860,KC,A,A,ARCH,OTH,Fall 2022,JR,AED 22860,Other
3,3,Registered,AED,22860,KC,F,B,ARCH,ID,Fall 2022,SO,AED 22860,CAED
4,4,Registered,AED,22860,KC,B,B,ARCH,ID,Fall 2022,FR,AED 22860,CAED
...,...,...,...,...,...,...,...,...,...,...,...,...,...
85336,85336,Registered,MDJ,20288,KC,B+,B,MDJ,FM,Spring 2023,SO,MDJ 20288,CotA
85337,85337,Registered,MDJ,20288,KC,A,A,MDJ,FM,Spring 2023,JR,MDJ 20288,CotA
85338,85338,Admin Dropped,MDJ,20288,KC,NaN,NaN,MDJ,COMM,Spring 2023,JR,MDJ 20288,CCI
85339,85339,Registered,MDJ,20288,KC,B+,B+,MDJ,FM,Spring 2023,SO,MDJ 20288,CotA


**Withdrawals and Mid Term Grades**

Fun fact about the system - if you withdraw before a mid term grade is posted, it does not enter a W in the Mid Term spot. It leaves it blank instead. This makes sense from a technical standpoint - W is not a valid mid term grade, so why enter it? However, when you want to see just how many people withdrew from the course by midterms, having a dedicated grade can help track that. As such, I will need to write out a statement that explicitly says "Hey - if they withdrew *and* no grade has been entered for mid terms, put a W down."

After some tinkering and note reviews, I realized that making a column that checks if a final grade of W was entered by no Mid Term grade would work as a first step. Then, reading that, making another statement that reads the results of that column to fill in the W.

In [302]:
#Withdrew W/o MT Grade Entered Pt. 1 (Oops 1)
#mtfull["Withdrew No MT"] = np.where(mtfull["Final Grade"] == "W" + mtfull["Mid Term Grade"].isna(), True, False) #I thought this would work because it's combining them - Let's try an &
#mtfull

In [303]:
#Withdrew W/o MT Grade Entered Pt. 2 (Oops 2)
#mtfull["Withdrew No MT"] = np.where(mtfull["Final Grade"] == "W" & mtfull["Mid Term Grade"].isna(), True, False) #didn't work again - going to try wrapping with () to see if that wakes it up

So the reason the above didn't work is due to a lack of understanding about symbols and pandas. The first time I thought a + would work, but I need an & to verbally say and in pandas speak. Then wrapping it with () helps identify them as separate conditions - doing them together makes an amalgamation that says both need to be true in an unreadable way.

In [304]:
#Withdrew W/o MT Grade Entered Pt. 3 (Making a new column - The Right Code)
mtfull["Withdrew No MT"] = np.where((mtfull["Final Grade"] == "W") & (mtfull["Mid Term Grade"].isna()), True, False) #It worked this time - huzzah
mtfull

,Record ID,Registration Status,Subject,Course,Campus,Final Grade,Mid Term Grade,Department,Major,Academic Period,Class,Course Code,College,Withdrew No MT
0,0,Registered,AED,22860,KC,A-,B,ARCH,ID,Fall 2022,FR,AED 22860,CAED,False
1,1,Registered,AED,22860,KC,A,A,ARCH,ID,Fall 2022,FR,AED 22860,CAED,False
2,2,Registered,AED,22860,KC,A,A,ARCH,OTH,Fall 2022,JR,AED 22860,Other,False
3,3,Registered,AED,22860,KC,F,B,ARCH,ID,Fall 2022,SO,AED 22860,CAED,False
4,4,Registered,AED,22860,KC,B,B,ARCH,ID,Fall 2022,FR,AED 22860,CAED,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
85336,85336,Registered,MDJ,20288,KC,B+,B,MDJ,FM,Spring 2023,SO,MDJ 20288,CotA,False
85337,85337,Registered,MDJ,20288,KC,A,A,MDJ,FM,Spring 2023,JR,MDJ 20288,CotA,False
85338,85338,Admin Dropped,MDJ,20288,KC,NaN,NaN,MDJ,COMM,Spring 2023,JR,MDJ 20288,CCI,False
85339,85339,Registered,MDJ,20288,KC,B+,B+,MDJ,FM,Spring 2023,SO,MDJ 20288,CotA,False


I did want to check if this worked and how many records there were - the numbers line up with what I normally see in this kind of data, so it didn't set anything off in my mind.

In [305]:
#Withdrew W/o MT Grade Entered Pt. 4 (Counting the Instances)
mtfull.groupby("Withdrew No MT").count()["Record ID"]

Withdrew No MT
False    83963
True      1378
Name: Record ID, dtype: int64

In [306]:
#Withdrew W/o MT Grade Entered Pt. 5 (Adding the Ws)
mtfull["Mid Term Grade"] = np.where(mtfull["Withdrew No MT"] == True, "W", mtfull["Mid Term Grade"])
mtfull["Mid Term Grade"].unique()

array(['B', 'A', nan, 'A-', 'C', 'C+', 'D', 'B+', 'B-', 'C-', 'F', 'S',
       'D+', 'W', 'U', 'SF', 'NF'], dtype=object)

In hindsight, making a single statement that replaces the Mid Term grade with a W when the original conditions were met would've been cleaner. Definitely something I want to consider when writing a "final" version of this code!

**Identifying Dropped Grades**

Similar to the withdrawal description above, drops do not get a grade at all. This makes sense since they don't get a grade when you drop the course. The logic below is most of the same and, since I already tackled the symbology issue above, I recalled | is used in place of or.

In [307]:
#Adding a DR Grade Pt. 1 (Writing the conditional column)
mtfull["Dropped no MT"] = np.where(((mtfull["Registration Status"] == "Std Dropped") | (mtfull["Registration Status"] == "Admin Dropped")) & (mtfull["Mid Term Grade"].isna()), True, False)

In [308]:
#Adding a DR Grade Pt. 2 (Counting how many instances)
mtfull.groupby("Dropped no MT").count()["Record ID"]

Dropped no MT
False    83989
True      1352
Name: Record ID, dtype: int64

In [309]:
#Adding a DR Grade Pt. 3 (Plugging in the DR values)
mtfull["Mid Term Grade"] = np.where(mtfull["Dropped no MT"] == True, "DR", mtfull["Mid Term Grade"])
mtfull["Mid Term Grade"].unique()

array(['B', 'A', nan, 'A-', 'C', 'C+', 'D', 'B+', 'B-', 'C-', 'F', 'S',
       'D+', 'DR', 'W', 'U', 'SF', 'NF'], dtype=object)

**Removing X Grades**

So this is a quirk of the data being reviewed well after the terms have concluded. Students are able to ask for grades to be expunged from their records after the term has ended if they meet extenuating circumstances. However, to indicate they still got a grade in the course, an X is added to before the grade assigned. In order to replicate how a review of this process would look when comparing grades across terms, I would want to remove that X from the grade. After some looking, using lstrip seems like the best way - it removes characters from a value (if present). This helped revert the values quickly.

I will note that this is a step that gets removed in the "final" version of this code, as we will be reviewing the grades shortly after posting and, in many cases, X grades do not start appearing well after the term has concluded.

In [310]:
#Removing X grades Pt. 1 (Checking for the X grade presence)
mtfull["Final Grade"].unique()

array(['A-', 'A', 'F', 'B', nan, 'D+', 'C', 'B-', 'B+', 'C-', 'W', 'C+',
       'S', 'D', 'XD', 'XF', 'SF', 'NF', 'AU', 'XSF', 'XC-', 'IN', 'XD+',
       'NR', 'XNF', 'U'], dtype=object)

In [311]:
#Removing X grades Pt. 2 (Removing and Checking for the grade)
mtfull["Final Grade"] = mtfull["Final Grade"].str.lstrip("X")
mtfull["Mid Term Grade"].unique()

array(['B', 'A', nan, 'A-', 'C', 'C+', 'D', 'B+', 'B-', 'C-', 'F', 'S',
       'D+', 'DR', 'W', 'U', 'SF', 'NF'], dtype=object)

**Adding a DR to Final Grades**

Replicating the DR process above, this time we are adding DRs to the final grades column. This will help minimize the presense of NaN values in that column as well.

In [312]:
#Adding DR to final grades Pt. 1 (Making the conditional column)
mtfull["Dropped no Fin"] = np.where(((mtfull["Registration Status"] == "Std Dropped") | (mtfull["Registration Status"] == "Admin Dropped")) & (mtfull["Final Grade"].isna()), True, False)
mtfull.groupby("Dropped no Fin").count()["Record ID"]

Dropped no Fin
False    83961
True      1380
Name: Record ID, dtype: int64

In [313]:
#Adding DR to final grades Pt. 2 (Adding the DR Grade)
mtfull["Final Grade"] = np.where(mtfull["Dropped no Fin"] == True, "DR", mtfull["Final Grade"])
mtfull["Final Grade"].unique()

array(['A-', 'A', 'F', 'B', nan, 'D+', 'C', 'B-', 'B+', 'C-', 'W', 'C+',
       'S', 'D', 'DR', 'SF', 'NF', 'AU', 'IN', 'NR', 'U'], dtype=object)

**Discrepancies between MT and Final Grades**

Something that set alarms off in my head was the difference in "True" values between the Final Grades DR and the Mid Term Grades DR values. After thinking about it, I realized it may be due to non-student drops, which can occur *after* the mid term grades are posted. Again, students can request late drops or expungements from their records, which could result in wonky grades. I want to confirm this, so I am going to look at records that have a Final Grade of DR and a Mid Term grade that isn't a DR.

In [314]:
#Checking disc between MT and final grades Pt. 1 (Looking at the instances)
mtfull.loc[(mtfull["Final Grade"] == "DR") & (mtfull["Mid Term Grade"] != "DR")]

,Record ID,Registration Status,Subject,Course,Campus,Final Grade,Mid Term Grade,Department,Major,Academic Period,Class,Course Code,College,Withdrew No MT,Dropped no MT,Dropped no Fin
2384,2384,Admin Dropped,ARCH,16841,KC,DR,SF,ARCH,FM,Fall 2024,FR,ARCH 16841,CotA,False,False,True
5169,5169,Admin Dropped,ARCH,11467,KC,DR,NF,ARCH,ARCH,Fall 2023,FR,ARCH 11467,CAED,False,False,True
6249,6249,Admin Dropped,ARCH,15217,KC,DR,NF,ARCH,OTH,Spring 2025,FR,ARCH 15217,Other,False,False,True
15281,15281,Admin Dropped,ARTH,17400,KC,DR,NF,ART,FD,Fall 2024,SR,ARTH 17400,CotA,False,False,True
21320,21320,Admin Dropped,CMGT,21745,KC,DR,B+,ARCH,ARCH,Spring 2023,JR,CMGT 21745,CAED,False,False,True
21346,21346,Admin Dropped,CMGT,21745,KC,DR,NF,ARCH,COMA,Spring 2024,SO,CMGT 21745,CAED,False,False,True
21505,21505,Admin Dropped,CMGT,21745,KC,DR,B,ARCH,ARCH,Spring 2025,JR,CMGT 21745,CAED,False,False,True
23375,23375,Admin Dropped,COMM,17591,KC,DR,NF,COMM,OTH,Spring 2023,FR,COMM 17591,Other,False,False,True
25210,25210,Admin Dropped,COMM,17591,KC,DR,NF,COMM,ADV,Spring 2024,FR,COMM 17591,CCI,False,False,True
25910,25910,Admin Dropped,COMM,17591,KC,DR,NF,COMM,VCD,Fall 2024,SR,COMM 17591,CCI,False,False,True


Looking at the list above, my read was right - the discrepancies were due to admin drops. Students most often get this when there are extenuating circumstances (i.e. hospitalizations) where they couldn't finish in time or didn't realize they were registered for classes and dropped on time. Since a midterm grade was still assigned, it would still reflect in the system. However, I want to makes sure I am catching the difference between the two, so I'm going to run a count.

In [315]:
#Checking disc between MT and final grades Pt. 2 (Checking the count)
mtfull.loc[(mtfull["Final Grade"] == "DR") & (mtfull["Mid Term Grade"] != "DR")].count()

Record ID              28
Registration Status    28
Subject                28
Course                 28
Campus                 28
Final Grade            28
Mid Term Grade         28
Department             28
Major                  28
Academic Period        28
Class                  28
Course Code            28
College                28
Withdrew No MT         28
Dropped no MT          28
Dropped no Fin         28
dtype: int64

Yep - that's the discrepancy! Since it can be explained, I am going to keep it as is. Again - we are trying to consider this as data viewed "in the moment" of what advisors would have seen - removing it would not be accurate to what the situation was when advisors would have been intervening. These interventions could also be why students were able to do a late drop - advisors contacted them, realized the situation, and started the paperwork to get a late drop done. I'm also going to extract the results up to this point - I am going to make a CSV file with this data for reference!

In [316]:
mtfull.to_csv("mtfullclean.csv")

**Removing the OTH Values**

OK - now that we have done all the manipulation needed, the mtfull variable can be used to help analyze the grades earned by *all* students in the classes above. Now we need to move into the intervention space, which would only impact students in the affected major(s). As such, we will make a variable that removes the OTH majors.

In [317]:
#Removing OTH Majors
mthubonly = mtfull[mtfull["Major"] != "OTH"].copy()
mthubonly

,Record ID,Registration Status,Subject,Course,Campus,Final Grade,Mid Term Grade,Department,Major,Academic Period,Class,Course Code,College,Withdrew No MT,Dropped no MT,Dropped no Fin
0,0,Registered,AED,22860,KC,A-,B,ARCH,ID,Fall 2022,FR,AED 22860,CAED,False,False,False
1,1,Registered,AED,22860,KC,A,A,ARCH,ID,Fall 2022,FR,AED 22860,CAED,False,False,False
3,3,Registered,AED,22860,KC,F,B,ARCH,ID,Fall 2022,SO,AED 22860,CAED,False,False,False
4,4,Registered,AED,22860,KC,B,B,ARCH,ID,Fall 2022,FR,AED 22860,CAED,False,False,False
5,5,Registered,AED,22860,KC,B,NaN,ARCH,ID,Fall 2022,FR,AED 22860,CAED,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
85336,85336,Registered,MDJ,20288,KC,B+,B,MDJ,FM,Spring 2023,SO,MDJ 20288,CotA,False,False,False
85337,85337,Registered,MDJ,20288,KC,A,A,MDJ,FM,Spring 2023,JR,MDJ 20288,CotA,False,False,False
85338,85338,Admin Dropped,MDJ,20288,KC,DR,DR,MDJ,COMM,Spring 2023,JR,MDJ 20288,CCI,False,True,True
85339,85339,Registered,MDJ,20288,KC,B+,B+,MDJ,FM,Spring 2023,SO,MDJ 20288,CotA,False,False,False


**Making the Intervention Column**

OK - now we get to the most complex part of the whole script. We need to make the different conditions for students to qualify for interventions from advisors. These conditions are set by the college, so I am going to use their conditions:
- Before the current term, FR/SO students had to get a C or below to qualify for an intervention, while JR/SR needed an F to qualify.
- Now, FR/SO students need a D+ or below to qualify. JR/SR still only need an F.
- We also need to consider dropped students, withdrawals, and NF grades - all of those students would not qualify since they are no longer registered for the course. SF grades still count, since the student attended at least some of the course and, if they return, maybe could finish the course and get a passing grade.
- We also need to differentiate between FR/SO and JR/SR students, so we need a condition for them too.
- We also also have to define the pre-Spring 2026 terms and the post-Spring 2026 terms.

With all that said - I found the best way to proceed is to define the variables first, then the conditions. There are a lot of variables for each category, followed by specific conditions. The FR/SO conditions are the most dense, followed by the JR/SR, and then the ones just checking their status.

In [318]:
#Making the Intervention Column Pt. 1 (Quick Reference for Grades)
mthubonly["Mid Term Grade"].unique()

array(['B', 'A', nan, 'A-', 'C', 'C+', 'D', 'B+', 'B-', 'C-', 'F', 'S',
       'D+', 'DR', 'W', 'SF', 'NF', 'U'], dtype=object)

In [319]:
#Making the Intervention Column Pt. 2 (Making all the variables)
pre202610frsopassgrade = ["A", "A-", "B+","B","B-","C+","S"]
pre202610frsofailgrade = ["C", "C-", "D+", "D", "F","U","SF"]
post202610frsopassgrade = ["A", "A-", "B+","B","B-","C+","C", "C-","S"]
post202610frsofailgrade = ["D+", "D", "F","U", "SF"]
jrsrpassgrade = ["A", "A-", "B+","B","B-","C+","C","C-","D+","D","S"]
jrsrfailgrade = ["F","U","SF"]
dropgrade = ["DR"]
wgrade = ["W"]
nfgrade = ["NF"]
frsocheck = ["FR","SO"]
jrsrcheck = ["JR","SR"]
pre202610terms = ["Fall 2022", "Spring 2023", "Fall 2023", "Spring 2024", "Fall 2024", "Spring 2025", "Fall 2025"]
post202610terms = ["Spring 2026"]

In [320]:
#Making the Intervention Column Pt. 3 (Making all the conditions and outcomes)
intercond = [
    ((mthubonly["Mid Term Grade"].isin(pre202610frsopassgrade)) & (mthubonly["Class"].isin(frsocheck)) & (mthubonly["Academic Period"].isin(pre202610terms))),
    ((mthubonly["Mid Term Grade"].isin(pre202610frsofailgrade)) & (mthubonly["Class"].isin(frsocheck)) & (mthubonly["Academic Period"].isin(pre202610terms))),
    ((mthubonly["Mid Term Grade"].isin(post202610frsopassgrade)) & (mthubonly["Class"].isin(frsocheck))  & (mthubonly["Academic Period"].isin(post202610terms))),
    ((mthubonly["Mid Term Grade"].isin(post202610frsofailgrade)) & (mthubonly["Class"].isin(frsocheck))  & (mthubonly["Academic Period"].isin(post202610terms))),
    ((mthubonly["Mid Term Grade"].isin(jrsrpassgrade)) & (mthubonly["Class"].isin(jrsrcheck))),
    ((mthubonly["Mid Term Grade"].isin(jrsrfailgrade)) & (mthubonly["Class"].isin(jrsrcheck))),
    mthubonly["Mid Term Grade"].isin(dropgrade),
    mthubonly["Mid Term Grade"].isin(wgrade),
    mthubonly["Mid Term Grade"].isin(nfgrade)
]

interout = ["Passing MT Grade", "FR/SO Intervention Needed", "Passing MT Grade", "FR/SO Intervention Needed", "Passing MT Grade", "JR/SR Intervention Needed", "Dropped Course", "Withdrew From Course", "Never Attended Course"]

In [321]:
#Making the Intervention Column Pt. 4 (Creating the Intervention column)
mthubonly["MT Grade Status"] = np.select(intercond,interout, "No MT Grade Entered") #Remembering that the last bit is the "Catch all" value, I chose to make it the No MT Grade Entered
mthubonly.head()

,Record ID,Registration Status,Subject,Course,Campus,Final Grade,Mid Term Grade,Department,Major,Academic Period,Class,Course Code,College,Withdrew No MT,Dropped no MT,Dropped no Fin,MT Grade Status
0,0,Registered,AED,22860,KC,A-,B,ARCH,ID,Fall 2022,FR,AED 22860,CAED,False,False,False,Passing MT Grade
1,1,Registered,AED,22860,KC,A,A,ARCH,ID,Fall 2022,FR,AED 22860,CAED,False,False,False,Passing MT Grade
3,3,Registered,AED,22860,KC,F,B,ARCH,ID,Fall 2022,SO,AED 22860,CAED,False,False,False,Passing MT Grade
4,4,Registered,AED,22860,KC,B,B,ARCH,ID,Fall 2022,FR,AED 22860,CAED,False,False,False,Passing MT Grade
5,5,Registered,AED,22860,KC,B,NaN,ARCH,ID,Fall 2022,FR,AED 22860,CAED,False,False,False,No MT Grade Entered


In [322]:
#Making the Intervention Column Pt. 5 (Checking the Values)
mthubonly.groupby("MT Grade Status").count()["Record ID"]

MT Grade Status
Dropped Course                 927
FR/SO Intervention Needed     5460
JR/SR Intervention Needed      652
Never Attended Course           82
No MT Grade Entered           5697
Passing MT Grade             45021
Withdrew From Course           948
Name: Record ID, dtype: int64

In order to confirm the No MT Grade values are accurate, I ran another report that checked if the values are true. When I did, I found that there were five values that were not NA. This tells me two things:
1. I missed something in my conditions earlier.
2. I need to see what those values are to see if they are true or not.

In [323]:
#Making the Intervention Column Pt. 5 (Checking the No MT Grade Values)
mthubonly.loc[(mthubonly["MT Grade Status"] == "No MT Grade Entered") & (mthubonly["Mid Term Grade"].isna())]

,Record ID,Registration Status,Subject,Course,Campus,Final Grade,Mid Term Grade,Department,Major,Academic Period,Class,Course Code,College,Withdrew No MT,Dropped no MT,Dropped no Fin,MT Grade Status
5,5,Registered,AED,22860,KC,B,NaN,ARCH,ID,Fall 2022,FR,AED 22860,CAED,False,False,False,No MT Grade Entered
40,40,Registered,AED,22860,KC,B,NaN,ARCH,ID,Fall 2022,JR,AED 22860,CAED,False,False,False,No MT Grade Entered
45,45,Registered,AED,22860,KC,B-,NaN,ARCH,ID,Fall 2022,FR,AED 22860,CAED,False,False,False,No MT Grade Entered
46,46,Registered,AED,22860,KC,A,NaN,ARCH,ID,Fall 2022,FR,AED 22860,CAED,False,False,False,No MT Grade Entered
50,50,Registered,AED,22860,KC,A,NaN,ARCH,ID,Fall 2022,FR,AED 22860,CAED,False,False,False,No MT Grade Entered
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
85294,85294,Registered,MDJ,20288,KC,C+,NaN,MDJ,FM,Spring 2023,SR,MDJ 20288,CotA,False,False,False,No MT Grade Entered
85304,85304,Std Withdrawn,MDJ,20288,KC,A,NaN,MDJ,COMM,Spring 2023,SO,MDJ 20288,CCI,False,False,False,No MT Grade Entered
85319,85319,Registered,MDJ,20288,KC,A,NaN,MDJ,FM,Spring 2023,SO,MDJ 20288,CotA,False,False,False,No MT Grade Entered
85334,85334,Registered,MDJ,20288,KC,A,NaN,MDJ,FM,Spring 2023,SO,MDJ 20288,CotA,False,False,False,No MT Grade Entered


In [324]:
#Making the Intervention Column Pt. 6 (Checking the No MT Grades w/ to see what is going on)
mthubonly.loc[(mthubonly["MT Grade Status"] == "No MT Grade Entered") & (mthubonly["Mid Term Grade"].notna())]

,Record ID,Registration Status,Subject,Course,Campus,Final Grade,Mid Term Grade,Department,Major,Academic Period,Class,Course Code,College,Withdrew No MT,Dropped no MT,Dropped no Fin,MT Grade Status
19608,19608,Registered,CMGT,16841,KC,D,A-,ARCH,COMA,Fall 2022,GR,CMGT 16841,CAED,False,False,False,No MT Grade Entered
20786,20786,Registered,CMGT,17964,KC,NaN,A,ARCH,COMA,Spring 2026,GR,CMGT 17964,CAED,False,False,False,No MT Grade Entered
20789,20789,Registered,CMGT,17964,KC,NaN,B+,ARCH,COMA,Spring 2026,GR,CMGT 17964,CAED,False,False,False,No MT Grade Entered
20879,20879,Registered,CMGT,19400,KC,A-,C+,ARCH,COMA,Spring 2023,GR,CMGT 19400,CAED,False,False,False,No MT Grade Entered
57847,57847,Registered,MDJ,25946,KC,NaN,C,MDJ,VCD,Spring 2026,GR,MDJ 25946,CCI,False,False,False,No MT Grade Entered


Well that's weird. The GR indicates those enrolled are graduate students. Graduate students are allowed to take undergraduate courses, but they tend to stick around in the upper level courses, not lower level courses. I did not foresee this happening - I should have checked the classes earlier to catch those graduate students.

Since the population of affected students is small, I am going to do a quick patch fix for them and overwrite the values with the "Passing MT Grades" value.

In [325]:
#Making the Intervention Column Pt. 7 (The Quick Fix)
mthubonly.loc[(mthubonly["MT Grade Status"] == "No MT Grade Entered") & (mthubonly["Mid Term Grade"].notna()),"MT Grade Status"] = "Passing MT Grade" 
mthubonly.groupby("MT Grade Status").count()["Record ID"]

MT Grade Status
Dropped Course                 927
FR/SO Intervention Needed     5460
JR/SR Intervention Needed      652
Never Attended Course           82
No MT Grade Entered           5692
Passing MT Grade             45026
Withdrew From Course           948
Name: Record ID, dtype: int64

**Creating the Cleaned File**

Now that the file has everything we need, we can extract it to a CSV file. This file will be handy to show the population of classes that have higher concentrations of interventions needed or student withdrawals, as well as instances where faculty are not entering midterm grades.

In [326]:
#Creating the cleaned file
mthubonly.to_csv("mthubonlyclean.csv")

**Making the Intervention File**

The next file we want to make is the file with the interventions only. This is what will be used to create the tables that are shown to leadership, as well as the files that advisors would reference when performing their outreach to students.

We only need students who need interventions. As such, we are going to make a version of the file with just the students marked as such. Students who got a passing grade or left the course are not included - the logic is that students are passing/no longer enrolled, so there is no need to contact them.

In [327]:
#Making the Intervention File Pt. 1 (Removing Non-Interventions)
mthubinterventions = mthubonly[((mthubonly["MT Grade Status"] == "FR/SO Intervention Needed") | (mthubonly["MT Grade Status"] == "JR/SR Intervention Needed"))].copy()
mthubinterventions

,Record ID,Registration Status,Subject,Course,Campus,Final Grade,Mid Term Grade,Department,Major,Academic Period,Class,Course Code,College,Withdrew No MT,Dropped no MT,Dropped no Fin,MT Grade Status
11,11,Registered,AED,22860,KC,NaN,C,ARCH,ID,Fall 2022,FR,AED 22860,CAED,False,False,False,FR/SO Intervention Needed
14,14,Registered,AED,22860,KC,C,D,ARCH,ID,Fall 2022,FR,AED 22860,CAED,False,False,False,FR/SO Intervention Needed
19,19,Registered,AED,22860,KC,B-,C,ARCH,ID,Fall 2022,SO,AED 22860,CAED,False,False,False,FR/SO Intervention Needed
31,31,Stopped Attending - Failed,AED,22860,KC,W,D,ARCH,ID,Fall 2022,SO,AED 22860,CAED,False,False,False,FR/SO Intervention Needed
33,33,Registered,AED,22860,KC,B-,C-,ARCH,ID,Fall 2022,FR,AED 22860,CAED,False,False,False,FR/SO Intervention Needed
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
85275,85275,Registered,MDJ,20288,KC,F,F,MDJ,FM,Fall 2022,SR,MDJ 20288,CotA,False,False,False,JR/SR Intervention Needed
85309,85309,Registered,MDJ,20288,KC,C-,F,MDJ,DMP,Spring 2023,JR,MDJ 20288,CCI,False,False,False,JR/SR Intervention Needed
85315,85315,Registered,MDJ,20288,KC,A-,D,MDJ,FM,Spring 2023,SO,MDJ 20288,CotA,False,False,False,FR/SO Intervention Needed
85316,85316,Registered,MDJ,20288,KC,C,C-,MDJ,FM,Spring 2023,SO,MDJ 20288,CotA,False,False,False,FR/SO Intervention Needed


A note on this part - at this point I realized that, when I shuffled the data, it mixed in values from the current term across the terms that already had existing data. This made it seem like the prior terms had instances where the instructor never entered a final grade when one should already exist. I had to go back to my dummy file and do some tweaking with it so the non-current term data would be shuffled amongst itself, while the current term data would just stay shuffled alone.

In [328]:
#Making the Intervention File Pt. 2 (Making the CSV file)
mthubinterventions.to_csv("mthubinterventions.csv")

**Reflecting on Milestone 1**

I had a lot of fun going through this! I've only done this process once in Excel, so it was interesting to think through how to apply the process and clean the data here. There are things that were easier to do - specifically the conditions. I struggled at first with writing the conditions. I had to sit and think through the different circumstances that would need to be accounted for, especially since we are looking at historical data that changed over time. Thankfully there were not many changes between terms, with the first being this term. I found writing them out on paper first helped, and then figuring out what combinations of values would be needed to make them accurate.

Looking ahead, I think the next step would be to replicate the deliverables that leadership receives highlighting the midterm pass rates for students. This would use the myfullclean data from earlier. This helps give leadership an idea of how all students in a course are doing over time. This will likely take the form of a filterable table that highlights the course code, the number of students who did not pass, who did pass, and the number of empty MT grade entries. I am unsure if this would also need a figure (like a pie chart), but that is something I am open to exploring! I would also like to make something that provides a breakdown of how many interventions are needed in a given term - this would be a deliverable for advisors that can serve as a reference document when performing outreach efforts.

I am also thinking of what may need added to this code when it moves into the "live" stage. The main thing I think of is how some courses have pre-reqs of earning a certain grade before qualifying fora  future course. I would need to add lines of code to the Intervention column stage that calls out those specific courses. I can't add it to this version though - I randomized the numbers so I couldn't identify those courses :(. There are also some quirks of the university's data system that need considered too - like how the report only shows student's primary majors. If a student has a secondary major that requires the course, it wouldn't be caught in the report. I am getting training in the reporting system to make my own reports, so I could always make my own version of the report that flips the majors if the secondary one is a major we are concerned with! I'm thinking this is where the OTH coding may be helpful - I could specify if primary major = OTH, and secondary doesn't, flip them. An even more complex thing to consider is specifying what majors require what courses - the current iteration just looks for students who are in the college **and** not doing well in midterms, not if they need the course for the major. I think this is OK though - not doing well in any course can negatively impact aid or GPA, so reaching out to those students we have some relationship with helps either way.

In any case - the path forward is clear and I am excited to take the next steps! I am thinking the deliverables to Milestone 2 may be the following (please feel free to recommend alternatives if fit!):
- Filterable table showing percentage of students who passed midterms
- Filterable figure (pie chart?) that provides a visualization of the above information
- Filterable table that shows count of who does v. does not need interventions
- Table that shows information of students who need contacted